# ClearSight V5: Professional Re-Identification (ReID) Pipeline

This notebook demonstrates the **Two-Pass Auto-Calibration** architecture. Because every CCTV video has different lighting and blur, the AI cannot use a hardcoded similarity threshold.

- **Pass 1:** The AI scans the video, evaluates every human body, and mathematically auto-calibrates the perfect Cosine Similarity threshold.
- **Pass 2:** The AI renders the final tracked video using the auto-calibrated target ID.

In [1]:
import cv2
import torch
import numpy as np
from torchvision import transforms
import torch.nn.functional as F
import facenet_pytorch
from PIL import Image
from ultralytics import YOLO
import matplotlib.pyplot as plt
from IPython.display import Video

c:\Users\rajti\raj123\envs\clearsight\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Initialize PyTorch AI Engines

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")

# 1. Identity Verification Model (FaceNet + MTCNN)
face_detector = facenet_pytorch.MTCNN(thresholds=[0.6, 0.7, 0.7], keep_all=True, device=device, min_face_size=10)
resnet = facenet_pytorch.InceptionResnetV1(pretrained='vggface2').eval().to(device)
preprocess = transforms.Compose([
    transforms.ToPILImage(), transforms.Resize((160, 160)),
    transforms.ToTensor(), transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# 2. Body Tracking Model (YOLOv8)
yolo_model = YOLO('yolov8n.pt')

Running on: cuda


### 2. Generate Master Vector (Target Identity)
Here we extract the facial embeddings from the target's photos.

In [3]:
def get_raw_face(img_rgb, box):
    x1, y1, x2, y2 = box.astype(int)
    w, h = x2 - x1, y2 - y1
    x, y = max(0, x1), max(0, y1)
    face_crop = img_rgb[y:y+h, x:x+w]
    if face_crop.size == 0: return None
    return face_crop

# You can change these to SRK or Messi depending on what you want to test
target_image_paths = [
    'C:/Users/rajti/Downloads/srk1.webp',
    'C:/Users/rajti/Downloads/srk2.jpg',
    'C:/Users/rajti/Downloads/srk3.jpg'
]

anchor_embeddings = []
for p in target_image_paths:
    img = Image.open(p).convert('RGB')
    img_rgb = np.array(img)
    boxes, probs = face_detector.detect(img_rgb)
    if boxes is not None and len(boxes) > 0:
        biggest_box = max(boxes, key=lambda b: (b[2]-b[0]) * (b[3]-b[1]))
        perfect_face = get_raw_face(img_rgb, biggest_box)
        tensor = preprocess(perfect_face).unsqueeze(0).to(device)
        with torch.no_grad():
            anchor_embeddings.append(resnet(tensor))
            print(f"Processed {p}")

master_embedding = torch.mean(torch.cat(anchor_embeddings), dim=0, keepdim=True)
print("Master Vector ready.")

Processed C:/Users/rajti/Downloads/srk1.webp
Processed C:/Users/rajti/Downloads/srk2.jpg
Processed C:/Users/rajti/Downloads/srk3.jpg
Master Vector ready.


### 3. Pass 1: Auto-Calibration Phase (Hyper-Scan)
Instead of guessing a threshold, the AI scans the first 5 seconds of the video, scores every single human body, and finds the true target mathematically.

In [4]:
video_path = 'C:/Users/rajti/Downloads/footage1.mp4'

TARGET_TRACK_ID = None
id_max_scores = {}

# Run YOLO on the first 150 frames
results_pass1 = yolo_model.track(source=video_path, classes=[0], stream=True, persist=True, verbose=False)

print("Running Pass 1: Auto-Calibration Hyper-Scan...")
frame_p1 = 0
for r in results_pass1:
    frame_p1 += 1
    if frame_p1 > 150:
        break # Only need first 5 seconds to calibrate
        
    frame_bgr = r.orig_img
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    
    if r.boxes.id is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        track_ids = r.boxes.id.cpu().numpy().astype(int)
        
        for box, track_id in zip(boxes, track_ids):
            x1, y1, x2, y2 = box.astype(int)
            x, y = max(0, x1), max(0, y1)
            w, h = x2 - x1, y2 - y1
            
            # Expand crop to ensure head isn't cut off by YOLO
            y_exp = max(0, int(y - h * 0.2))
            body_crop = frame_rgb[y_exp:y+h, x:x+w]
            if body_crop.size == 0: continue
            
            face_boxes, probs = face_detector.detect(body_crop)
            if face_boxes is not None and len(face_boxes) > 0:
                biggest_face = max(face_boxes, key=lambda b: (b[2]-b[0]) * (b[3]-b[1]))
                perfect_face = get_raw_face(body_crop, biggest_face)
                if perfect_face is not None:
                    with torch.no_grad():
                        tensor = preprocess(perfect_face).unsqueeze(0).to(device)
                        embedding = resnet(tensor)
                    
                    cosine_sim = F.cosine_similarity(master_embedding, embedding).item()
                    if track_id not in id_max_scores or cosine_sim > id_max_scores[track_id]:
                        id_max_scores[track_id] = cosine_sim

if id_max_scores:
    best_id = max(id_max_scores, key=id_max_scores.get)
    best_score = id_max_scores[best_id]
    if best_score > 0.12:
        TARGET_TRACK_ID = best_id
        print(f"🎯 AUTO-CALIBRATION SUCCESS: Locked onto Body #{TARGET_TRACK_ID} (Peak Confidence: {best_score:.3f})")
    else:
        print(f"⚠️ FAILED: No person matched the Master Vector.")
else:
    print("⚠️ FAILED: No faces detected in the scanning phase.")

Running Pass 1: Auto-Calibration Hyper-Scan...
🎯 AUTO-CALIBRATION SUCCESS: Locked onto Body #12 (Peak Confidence: 0.351)


### 4. Pass 2: Rendering Phase
Now that the AI mathematically knows the exact Body ID of the target, it rewinds the video and permanently locks onto that body ID.

In [5]:
if TARGET_TRACK_ID is not None:
    out_path = 'tracked_footage1_auto_calibrated.mp4'
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

    print("Running Pass 2: Final Video Render...")
    
    # Reload YOLO model to reset tracker history
    yolo_model = YOLO('yolov8n.pt')
    results_pass2 = yolo_model.track(source=video_path, classes=[0], stream=True, persist=True, verbose=False)

    for r in results_pass2:
        frame_bgr = r.orig_img
        
        if r.boxes.id is not None:
            boxes = r.boxes.xyxy.cpu().numpy()
            track_ids = r.boxes.id.cpu().numpy().astype(int)
            
            for box, track_id in zip(boxes, track_ids):
                if track_id == TARGET_TRACK_ID:
                    x1, y1, x2, y2 = box.astype(int)
                    cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 255, 0), 3)
                    cv2.putText(frame_bgr, f"LOCKED RE-ID [{track_id}]", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        
        out.write(frame_bgr)

    cap.release()
    out.release()
    print(f"Done! Video saved to {out_path}")
else:
    print("Cannot run Pass 2 because Pass 1 failed to lock a target.")

Running Pass 2: Final Video Render...
Done! Video saved to tracked_footage1_auto_calibrated.mp4
